# Mann-Whitney U Evaluation

In [ ]:

from pathlib import Path

import pandas as pd
from scipy.stats import mannwhitneyu

BASE_DIR = Path(".")
SAMPLE_SIZE = 335
RANDOM_STATE = 335

FILES = {
    "CodeLlama": BASE_DIR / "CodeLlama_Automatic_Evaluation_Results.xlsx",
    "DeepSeek": BASE_DIR / "DeepSeek_Automatic_Evaluation_Results.xlsx",
    "Qwen": BASE_DIR / "Qwen_Automatic_Evaluation_Results.xlsx",
}

STRATEGIES = {
    "Zero-shot": [
        "BLEU Zero",
        "ROUGE Zero",
        "METEOR Zero",
    ],
    "Few-shot": [
        "BLEU Few",
        "ROUGE Few",
        "METEOR Few",
    ],
    "Chain of Thought": [
        "BLEU Adv",
        "ROUGE Adv",
        "METEOR Adv",
    ],
}

COMPARISONS = [
    ("Zero-shot", "Few-shot"),
    ("Zero-shot", "Chain of Thought"),
    ("Few-shot", "Chain of Thought"),
]

frames = []

for model, file_path in FILES.items():
    frame = pd.read_excel(file_path)

    if "Skipped" in frame.columns:
        frame = frame[frame["Skipped"] == False].copy()

    if len(frame) < SAMPLE_SIZE:
        raise ValueError(
            f"{model} contains {len(frame)} evaluated rows; "
            f"{SAMPLE_SIZE} rows are required."
        )

    frame = frame.sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_STATE,
    )
    frame["Model"] = model
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

numeric_columns = [
    column
    for columns in STRATEGIES.values()
    for column in columns
]

for column in numeric_columns:
    df[column] = (
        df[column]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce",
    )

results = []

for model in FILES:
    model_df = df[df["Model"] == model]

    for metric_name, metric_index in zip(
        ["BLEU", "ROUGE", "METEOR"],
        [0, 1, 2],
    ):
        for strategy_1, strategy_2 in COMPARISONS:
            column_1 = STRATEGIES[strategy_1][metric_index]
            column_2 = STRATEGIES[strategy_2][metric_index]

            data_1 = model_df[column_1].dropna()
            data_2 = model_df[column_2].dropna()

            statistic, p_value = mannwhitneyu(
                data_1,
                data_2,
                alternative="two-sided",
            )

            mean_1 = data_1.mean()
            mean_2 = data_2.mean()

            results.append({
                "Model": model,
                "Metric": metric_name,
                "Comparison": f"{strategy_1} vs {strategy_2}",
                "Mean 1": round(mean_1, 4),
                "Mean 2": round(mean_2, 4),
                "U-Statistic": round(statistic, 4),
                "P-Value": p_value,
                "Significant": "Yes" if p_value < 0.05 else "No",
                "Higher Mean": strategy_1 if mean_1 > mean_2 else strategy_2,
            })

results_df = pd.DataFrame(results)
results_df.to_excel(
    "Mann_Whitney_U_Results.xlsx",
    index=False,
)
display(results_df)
